# Anchoring regression V4 (LEC) - past *and* future lagsPer-neuron regression of raw 25 ms firing onto (location x goal-progress x lag) anchors,leave-one-session-out, after El-Gaby et al. 2024 Figure 5. Module:[`elasticnet_regression_v4.py`](elasticnet_regression_v4.py); v3 is left untouched for diffing.**What V4 adds over V3*** `lag_direction='future'` - the prospective mirror of the reference's retrospective model  ("I will be at location 5 in 3 steps"), built by reversing the location/phase sequence  through the *identical* bump loop.* Per-neuron exports: betas (averaged **and** per fold), 360-bin actual/predicted tuning  curves, and the n=4 preferred-phase state vectors the headline r is computed from -  as `.npz` arrays plus paged PDFs, for **all** neurons and for the non-zero-lag subset.* The region join (`build_unit_table` / `region_summary`), with mean firing rate carried  along because at a fixed alpha the fit is partly a firing-rate cut.**Things to know before reading any output**1. Anchor phase is *determined* by preferred phase and lag (`ap == (pref -/+ lag) % 3`), so   only 108 of the 324 columns are live in any fit and the beta matrix is stored/plotted as   `location x lag`. The phase axis carries no independent information.2. The prediction is therefore **exactly zero** at every non-preferred-phase bin: 240 of the   360 bins of a "predicted tuning curve" are zero by construction. Use   `tuning_corr_pref` (preferred-phase bins only), not `tuning_corr` (all 360).3. Preferred phase is refit per fold and ~36% of neurons change it, so averaging betas across   folds superimposes two coordinate frames. The non-zero-lag criterion runs per fold, and the   `*_foldbetas.pdf` files show each fold separately.4. At `alpha_mode='fixed', elasticnet_alpha=0.01` on raw counts, ~61% of neurons fit all-zero   and the survivors are the fast-firing ones. `alpha_mode='relative'` puts every neuron at   the same point on its own regularization path. Check `recday_diagnostics_*.csv`.Gate: `python elasticnet_v4_synthetics.py` - 28 controls, including exact reproduction of v3.

In [ ]:
import numpy as npimport scipy.stats as stfrom scipy import statsfrom scipy.stats import zscorefrom scipy.ndimage import gaussian_filter1dimport matplotlib.pyplot as pltimport seaborn as snsimport pandas as pdimport os, picklefrom tqdm import tqdm

In [ ]:
Data_folder="../data/processed_data"figures_folder="../data/figures"neuron_folder = f"{Data_folder}/neuron_raw_mingyutest"trialtimes_folder = f"{Data_folder}/trialtimes_raw_mingyutest"tracking_folder = f"{Data_folder}/processed"

In [ ]:
import pickleimport os# Define the folder where the processed data is savedprocessed_data_folder = "../data/processed_data"# Load the main data dictionarywith open(os.path.join(processed_data_folder, 'data_dic_lec.pkl'), 'rb') as f:    data_dic = pickle.load(f)# Load the normalized neurons dictionarywith open(os.path.join(processed_data_folder, 'norm_neurons_dic.pkl'), 'rb') as f:    norm_neurons_dic = pickle.load(f)# Load the session indices dictionarywith open(os.path.join(processed_data_folder, 'session_inds_dic.pkl'), 'rb') as f:    session_inds_dic = pickle.load(f)# Load the tasks dictionarywith open(os.path.join(processed_data_folder, 'tasks_dic.pkl'), 'rb') as f:    tasks_dic = pickle.load(f)print(f"Loaded data dictionaries from {processed_data_folder}")

In [ ]:
all_files = os.listdir(neuron_folder)neuron_files = [f for f in all_files if f.startswith('Neuron_raw_') and f.endswith('.npy')]mouse_recdays_ = []for f in neuron_files:    # Remove prefix 'Neuron_raw_' and suffix '.npy'    stripped_name = f[len('Neuron_raw_'):-len('.npy')]    # Split by '_' and rejoin all parts except the last one (session number)    mouse_recday = '_'.join(stripped_name.split('_')[:-1])    mouse_recdays_.append(mouse_recday)# Find the unique mouse_recday identifiersmouse_recdays = np.unique(mouse_recdays_)# Filter out identifiers containing '_sb'unique_mouse_recdays = [recday for recday in mouse_recdays if '_sb' not in recday]print("Unique mouse_recday identifiers (without '_sb'):")print(unique_mouse_recdays)# Assign the first unique identifier to mouse_recday for later useif len(unique_mouse_recdays) > 0:    mouse_recday = unique_mouse_recdays[0]mouse_recdays = unique_mouse_recdays

In [ ]:
# Build valid_sessions_dic: for each mouse_recday, keep one session per unique task structurevalid_sessions_dic = {}for mouse_recday in mouse_recdays:    valid_sessions = []    tasks = []    for session in list(data_dic[mouse_recday].keys()):        if session == 'valid_sessions':            continue        if data_dic[mouse_recday][session]['num_trials'] < 5:            print(f'{mouse_recday} session {session}: not enough trials, skipping')            continue        if 'defaultdict' in str(data_dic[mouse_recday][session]['Task']):            print(f'{mouse_recday} session {session}: no task, skipping')            continue        if not any(np.array_equal(data_dic[mouse_recday][session]['Task'], candidate) for candidate in tasks):            tasks.append(data_dic[mouse_recday][session]['Task'])            valid_sessions.append(session)    print(f'{mouse_recday}: {valid_sessions}')    valid_sessions_dic[mouse_recday] = valid_sessions

## Run - past and future lagsBoth directions, same config otherwise. `n_jobs` uses joblib's threading backend, so `data_dic` is shared rather than copied to workers.

In [ ]:
import importlib, osfrom datetime import datetimeimport elasticnet_regression_v4 as v4importlib.reload(v4)STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')N_JOBS = 6                      # 8 cores on this box; threading, so memory is shareddef make_config(direction, **kw):    # El-Gaby's executed default is Poisson (use_poisson=True, alpha=1, no sparsification)    # but it is ~7x slower per fit. The ElasticNet branch below is the paper's stated one.    return v4.RegressionConfigV4(        use_poisson=False, regularize=True,      # ElasticNet, alpha=0.01, positive        lag_direction=direction,        alpha_mode='fixed',                      # 'relative' + alpha_frac to rescale per neuron        pref_phase_source='train',               # 'test' reproduces the reference's leakage        state_reduce='mean',                     # 'max' also stored either way        **kw)results, pooled, diagnostics = {}, {}, {}for direction in ('past', 'future'):    cfg = make_config(direction)    # Folder name leads with the estimator (poisson / elasticnet / linear) -- it is the    # setting that most changes the numbers. Everything else is in run_config.json,    # written into the same folder as the arrays and figures.    out_dir = os.path.join('../data/figures', v4.run_dir_name(cfg, stamp=STAMP))    print(f'\n{"="*70}\n{direction.upper()} lags -> {out_dir}\n{"="*70}')    results[direction], pooled[direction], diagnostics[direction] = v4.run_and_summarise_all_mice_v4(        data_dic, cfg,        valid_sessions_dic=valid_sessions_dic,        save_dir=out_dir, export_dir=out_dir,        make_pdfs=True, n_jobs=N_JOBS, verbose=True)configs = {d: make_config(d) for d in results}

### Per-recday diagnosticsRead this before the region table. `frac_allzero_fits` is the alpha dropout, `frac_pref_phase_flips` the fraction of neurons whose coordinate frame changes between folds, and `n_nonzero_lag_alt_top3` the mask size with the argsort tie-breaking guard flipped - a large gap there means the mask is partly sort-order artefact.

In [ ]:
for direction, tab in diagnostics.items():    print(f'\n===== {direction} =====')    print(tab.to_string(index=False))

### The state-tuning filter is confounded by leg duration - check this before trusting `selected``selected = nonzero_lag & state_tuned & mean_corr.notna()`, and the `state_tuned` term does notmean what it says. `raw_to_norm` warps each state interval onto 90 bins by *averaging*, so alonger leg puts more raw bins into each normalised bin, lowering its variance and therefore its**max** - which is the statistic the El-Gaby test z-scores across states. Constant-rate Poissoncells with no tuning at all then acquire a "preferred state": the shortest leg.Measured on 300 pure-noise cells per row, nominal alpha = 0.05:| longest:shortest leg | FPR (`max`) | prefer shortest leg | FPR (`mean`) ||---|---|---|---|| 1.0x | 0.053 | 26% (chance) | 0.097 || 2.0x | **0.970** | **94%** | 0.117 || 3.0x | **1.000** | **99%** | 0.073 |The median within-session longest:shortest mean leg duration **in this dataset is 2.26x**(p90 5.4x, max 11.5x, n=164 sessions). So at the real data's leg inequality the filter passesessentially everything and selects on leg geometry, not tuning.And it is measurably present in the real data: across 31 sessions (1,860 units), **47.4% ofreal units "prefer" the shortest leg** against a chance of 25%, and the per-session fractioncorrelates with that session's leg-duration ratio at **r = 0.43**. Real neurons do carrygenuine tuning - the effect is about half the pure-noise strength - but a large share of thepreferred-state assignment is leg geometry.`state_tuning_statistic='mean'` is duration-invariant and stays near nominal. The default stays`'max'` to match the reference - change it deliberately. Both masks are computed on every run(`state_tuned_mask` and `state_tuned_mask_alt`), so the comparison below needs no re-run.

In [ ]:
# How saturated is the tuning filter, and does the preferred state just track the shortest leg?for direction, tab in diagnostics.items():    print(f'\n===== {direction} =====')    print(tab[['mouse_recday', 'n_neurons', 'n_state_tuned', 'n_state_tuned_alt_stat',               'state_duration_ratio', 'frac_pref_state_is_shortest']].to_string(index=False))# The same region table under the duration-invariant statistic. If the anatomy result only# exists under 'max', it is a leg-geometry result.tab = tables['past'].copy()tab['selected'] = tab.nonzero_lag & tab.state_tuned_alt_stat & tab.mean_corr.notna()print('\n===== past lags, state tuning by MEAN (duration-invariant) =====')v4.region_summary(tab)

## Which brain regions do these neurons come from?

In [ ]:
regions = v4.load_unit_regions()tables = {d: v4.build_unit_table(results[d], configs[d], regions=regions, data_dic=data_dic)          for d in results}for direction, tab in tables.items():    print(f'\n{"="*70}\n{direction.upper()} lags - selection by region\n{"="*70}')    v4.region_summary(tab)

### Past vs future, per region`pro_index = (r_future - r_past) / (|r_future| + |r_past|)`: positive means a unit is betterexplained prospectively. The two designs are not degenerate - past lag *k* and future lag12-*k* point at the same task position one loop apart, and their measured column correlationis only ~0.02-0.34, because routes vary between trials.

In [ ]:
merged, direction_summary = v4.compare_directions(tables['past'], tables['future'])merged.head()

### The firing-rate confoundAt a fixed ElasticNet alpha a unit is only fittable if it fires fast enough (all-zero fitsaverage ~0.7 Hz, surviving fits ~6.4 Hz), so a region difference in *selection rate* can be aregion difference in *firing rate*. `region_summary` already prints the within-quartileversion above. To check how much of the effect is the penalty rather than the anatomy, re-runone recday with a per-neuron relative alpha and compare.

In [ ]:
# Rate-matched re-run of a single recday (cheap): every neuron sits at the same point on# its own regularization path instead of a shared absolute alpha.mr = list(results['past'].keys())[0]cfg_rel = make_config('past', alpha_mode='relative', alpha_frac=0.1)res_rel = v4.run_cross_validated_regression_v4(    data_dic, mr, cfg_rel, valid_sessions=valid_sessions_dic[mr], verbose=True)tab_rel = v4.build_unit_table({mr: res_rel}, cfg_rel, regions=regions, data_dic=data_dic)print('\n--- relative alpha ---')v4.region_summary(tab_rel)print('\n--- fixed alpha, same recday ---')v4.region_summary(tables['past'][tables['past'].recday == mr])

## Inspecting one neuron`fold_betas` returns each fold's beta matrix collapsed in *that fold's* frame - the un-averaged view of what the `*_foldbetas.pdf` pages show.

In [ ]:
mr = list(results['past'].keys())[0]res = results['past'][mr]top = np.argsort(np.nan_to_num(res['mean_tuning_correlations_pref'], nan=-np.inf))[::-1][:5]print('top neurons by preferred-phase tuning r:', top.tolist())ni = int(top[0])print(f'neuron {ni}: pref phase per fold = {res["pref_phases"][ni].tolist()}, '      f'peak lag = {res["peak_lags"][ni]}, r = {res["mean_corrs"][ni]:.3f}, '      f'non-zero betas/fold = {res["n_nonzero_betas"][ni].tolist()}')fb = v4.fold_betas(res, configs['past'], ni)          # (n_folds, 9 locations, 12 lags)fig, axes = plt.subplots(1, len(fb) + 1, figsize=(2.2 * (len(fb) + 1), 2.4))vmax = np.nanmax(np.abs(fb)) or 1.0for fi, ax in enumerate(axes[:-1]):    ax.imshow(fb[fi], aspect='auto', cmap='hot', vmin=0, vmax=vmax)    ax.set_title(f'fold {fi} (pref {res["pref_phases"][ni, fi]})', fontsize=7)    ax.tick_params(labelsize=5)B, modal, n_used, n_tot = v4.betas_in_common_frame(res, configs['past'], ni)axes[-1].imshow(B, aspect='auto', cmap='hot', vmin=0, vmax=vmax)axes[-1].set_title(f'mean [{n_used}/{n_tot} folds, pref {modal}]', fontsize=7)axes[-1].tick_params(labelsize=5)fig.suptitle(f'{mr} neuron {ni} - past lags (location x lag)', fontsize=9)fig.tight_layout()

## Reading the exports backOne `.npz` per recday per direction, plus the PDFs. Nothing here needs `data_dic`.

In [ ]:
import globout_dir = os.path.join('../data/figures',                       v4.run_dir_name(configs['past'], stamp=STAMP))print('\n'.join(sorted(os.path.basename(p) for p in glob.glob(out_dir + '/*'))[:12]))z = v4.load_regression_outputs(glob.glob(out_dir + '/*_arrays.npz')[0])for k in sorted(z):    print(f'  {k:32s} {getattr(z[k], "shape", type(z[k]).__name__)}')